<h1><center>Laboratorio 7: Ensamblaje, Optimización de Hiperparámetros e Interpretabilidad 🤖</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco y Ignacio Reyes

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Antonia Landaeta
- Nombre de alumno 2: Felipe Muñoz

---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Antigravity, Cursor, etc.) restringido a consultas, documentación y corrección de errores.

## Temas a tratar

- Ensamblaje: Bagging (`RandomForest`), Boosting (`XGBoost`, `LightGBM`) y Stacking.
- Optimización de Hiperparámetros con `Optuna` y visualización interactiva con `optuna-dashboard`.
- Interpretabilidad global: `Permutation Feature Importance (PFI)`.
- Interpretabilidad local: `SHAP`.

### Objetivos principales del laboratorio

- Aplicar y comparar métodos de ensamblaje sobre un problema de clasificación de texto.
- Optimizar hiperparámetros de LightGBM usando Optuna y visualizar el proceso con `optuna-dashboard`.
- Interpretar las predicciones del modelo usando PFI y SHAP.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de Python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`.

### Instalamos librerías 😸

In [ ]:
!uv add nltk lightgbm xgboost optuna shap scikit-learn plotly

In [1]:
import warnings

import matplotlib.pyplot as plt
import nltk
import numpy as np
import optuna
import pandas as pd
import shap
from lightgbm import LGBMClassifier
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split  # noqa: F401
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from optuna.visualization import (
    plot_parallel_coordinate,
    plot_optimization_history,
    plot_param_importances,
)
import plotly.express as px
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

RANDOM_STATE = 42
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. ¿Quién es Bat Cow?

<p align="center">
  <img src="https://i.imgur.com/D9f1RHy.jpg" width="350">
</p>

En vez de estar desarrollando las evaluaciones correspondientes a su curso, su profesor de catedra y su auxiliar discuten acerca la alineación (i.e., si es heroe o villano) del personaje de ficción Bat-Cow.

El cuerpo docente, no logra ponerse de acuerdo si el personaje es bueno, neutral o malo: el auxiliar plantea que Bat-cow posee una siniestra mirada, intrigante pero común característica de los personajes malvados.
Por otra parte, extendiendo las ideas de Rousseau, el profesor plantea que tal como los humanos no nacen malos, no existe motivo por el cual una vaca con superpoderes deba serlo.

Sin embargo, ambos concuerdan que es difícil estimar la alineación solo usando los atributos físicos. Es por esto que les solicitan construir y optimizar un clasificador basado en texto que analice la alineación de cada personaje basado en su historia personal.

Para este laboratorio deben trabajar con los datos `df_comics.csv` y `comics_no_label.csv` subidos a u-cursos.

In [2]:
df_comics = pd.read_csv("df_comics.csv", index_col=0)
df_comics_no_label = pd.read_csv("comics_no_label.csv", index_col=0)
df_comics = df_comics.dropna(subset=["history_text"])
df_comics

,name,real_name,full_name,overall_score,history_text,powers_text,intelligence_score,strength_score,speed_score,durability_score,...,has_flight,has_accelerated_healing,has_weapons_master,has_intelligence,has_reflexes,has_super_speed,has_durability,has_stamina,has_agility,has_super_strength
0,3-D Man,"Delroy Garrett, Jr.","Delroy Garrett, Jr.",6,"Delroy Garrett, Jr. grew up to become a track ...",NaN,85,30,60,60,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,A-Bomb,Richard Milhouse Jones,Richard Milhouse Jones,20,"Richard ""Rick"" Jones was orphaned at a young ...","On rare occasions, and through unusual circu...",80,100,80,100,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0
3,Aa,Aa,NaN,12,Aa is one of the more passive members of the P...,NaN,80,50,55,45,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Aaron Cash,Aaron Cash,Aaron Cash,5,Aaron Cash is the head of security at Arkham A...,NaN,80,10,25,40,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,Aayla Secura,Aayla Secura,NaN,8,ayla Secura was a Rutian Twi'lek Jedi Knight (...,NaN,90,40,45,55,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1445,Zatanna,Zatanna Zatara,Zatanna Zatara,10,Zatanna is the daughter of adventurer John Zat...,Zatanna is genetically talented with her magi...,90,10,25,30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1446,Zero,DWN-∞: Zero,DWN-∞: Zero,18,Zero was created by the late Dr. Albert Wily ...,NaN,80,100,100,100,...,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1447,Zoom (New 52),Hunter Zolomon,NaN,20,"Hunter Zolomon is better known as Zoom, a spee...",After tricking Barry Allen and Wally West into...,95,50,100,75,...,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1448,Zoom,Hunter Zolomon,Hunter Zolomon,9,Hunter Zolomon had a troubled relationship wi...,"Zoom is able to alter time, to make himself ev...",75,10,100,30,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [10]:
# imprimir en una lista el nombre de las columnas del dataframe
print(df_comics.columns.tolist())

['name', 'real_name', 'full_name', 'overall_score', 'history_text', 'powers_text', 'intelligence_score', 'strength_score', 'speed_score', 'durability_score', 'power_score', 'combat_score', 'superpowers', 'alter_egos', 'aliases', 'place_of_birth', 'first_appearance', 'creator', 'alignment', 'occupation', 'base', 'teams', 'relatives', 'gender', 'type_race', 'height', 'weight', 'eye_color', 'hair_color', 'skin_color', 'img', 'has_electrokinesis', 'has_energy_constructs', 'has_mind_control_resistance', 'has_matter_manipulation', 'has_telepathy_resistance', 'has_mind_control', 'has_enhanced_hearing', 'has_dimensional_travel', 'has_element_control', 'has_size_changing', 'has_fire_resistance', 'has_fire_control', 'has_dexterity', 'has_reality_warping', 'has_illusions', 'has_energy_beams', 'has_peak_human_condition', 'has_shapeshifting', 'has_heat_resistance', 'has_jump', 'has_self-sustenance', 'has_energy_absorption', 'has_cold_resistance', 'has_magic', 'has_telekinesis', 'has_toxin_and_disea

In [ ]:
# imprimir los valores únicos de la columna 'alignment'
df_comics['alignment'].unique()

<StringArray>
['Good', 'Bad', 'Neutral']
Length: 3, dtype: str

In [ ]:
# contar el número de ocurrencias de cada valor en la columna 'alignment'
df_comics['alignment'].value_counts()

alignment
Good       743
Bad        429
Neutral    113
Name: count, dtype: int64

`alignment` será usada como variable objetivo, pues indica la alineación moral de cada personaje: bueno, malo o neutral. Donde, un heroé podrá ser bueno o neutral, y un villano es clasificado como malo.

## 1.1 Obtención de Features y Bag of Words

<p align="center">
  <img src="https://media0.giphy.com/media/eIUpSyzwGp0YhAMTKr/200.gif" width="300">
</p>

`bag of words` es un modelo de conteo utilizado en NLP que genera una representación vectorial para cada documento a través del conteo de las palabras que contienen.

<p align="center">
  <img src="https://user.oc-static.com/upload/2020/10/23/16034397439042_surfin%20bird%20bow.png" width="500">
</p>

Para facilitar el conteo transformamos cada documento en un vector mediante **tokenización**:

In [3]:
docs = ["The teacher rocks like a good rock & roll", "the rock is the best actor in the world"]
docs_tokenizados = [word_tokenize(doc) for doc in docs]
docs_tokenizados

[['The', 'teacher', 'rocks', 'like', 'a', 'good', 'rock', '&', 'roll'],
 ['the', 'rock', 'is', 'the', 'best', 'actor', 'in', 'the', 'world']]

Podemos mejorar la tokenización con:

- **Stemming**: transforma palabras a su forma raíz (*running → run*, *rocks → rock*).
- **Eliminación de Stopwords**: elimina palabras muy frecuentes que entorpecen la clasificación (*the*, *is*, *a*, ...).

<p align="center">
  <img src="https://devopedia.org/images/article/218/8583.1569386710.png" width="300">
</p>

In [4]:
stop_words = stopwords.words("english")


class StemmerTokenizer:
    def __init__(self):
        self.ps = PorterStemmer()

    def __call__(self, doc):
        doc_tok = word_tokenize(doc)
        doc_tok = [t for t in doc_tok if t not in stop_words]
        return [self.ps.stem(t) for t in doc_tok]


tokenizador = StemmerTokenizer()

docs = [
    "The teacher rocks like a good rock & roll",
    "the rock is the best actor in the world",
    "New York is a beautiful city",
]

print("Con StemmerTokenizer:")
print([tokenizador(doc) for doc in docs])
print("\nSin preprocesamiento:")
print([word_tokenize(doc) for doc in docs])

Con StemmerTokenizer:
[['the', 'teacher', 'rock', 'like', 'good', 'rock', '&', 'roll'], ['rock', 'best', 'actor', 'world'], ['new', 'york', 'beauti', 'citi']]

Sin preprocesamiento:
[['The', 'teacher', 'rocks', 'like', 'a', 'good', 'rock', '&', 'roll'], ['the', 'rock', 'is', 'the', 'best', 'actor', 'in', 'the', 'world'], ['New', 'York', 'is', 'a', 'beautiful', 'city']]


#### Al Estilo Scikit

Scikit implementa `bag of words` con `CountVectorizer()`. Además soporta **n-gramas**: secuencias contiguas de n palabras que se tratan como un único token. Esto permite capturar contexto local que los unigramas pierden.

| Tipo | n | Tokens de `"nueva york ciudad"` |
|------|---|--------------------------------|
| Unigrama | 1 | `nueva`, `york`, `ciudad` |
| Bigrama | 2 | `nueva york`, `york ciudad` |
| Trigrama | 3 | `nueva york ciudad` |

Con `ngram_range=(1,2)` el vectorizador incluye **unigramas y bigramas** simultáneamente. Los bigramas son especialmente útiles para capturar expresiones compuestas como `bat cow`, `spider man` o `super hero` que pierden su significado si se separan.

El parámetro `max_features` limita el vocabulario a los n tokens más frecuentes, controlando la dimensionalidad de la representación.

In [5]:
bow = CountVectorizer(tokenizer=StemmerTokenizer(), ngram_range=(1, 2))
df_bow = bow.fit_transform(docs)
pd.DataFrame(df_bow.toarray(), columns=bow.get_feature_names_out())

,&,& roll,actor,actor world,beauti,beauti citi,best,best actor,citi,good,...,rock,rock &,rock best,rock like,roll,teacher,teacher rock,world,york,york beauti
0,1,1,0,0,0,0,0,0,0,1,...,2,1,0,1,1,1,1,0,0,0
1,0,0,1,1,0,0,1,1,0,0,...,1,0,1,0,0,0,0,1,0,0
2,0,0,0,0,1,1,0,0,1,0,...,0,0,0,0,0,0,0,0,1,1


#### Combinando Features: `ColumnTransformer`

Para combinar en un solo paso el preprocesamiento de texto y numérico, usamos `ColumnTransformer`. Este aplica transformadores distintos a subconjuntos de columnas del DataFrame y concatena el resultado en una sola matriz de features lista para entrenar.

<p align="center">
  <img src="https://c.tenor.com/LkQzw7k5DV4AAAAd/anime-hacking.gif" width="300">
</p>

El `preprocessing_transformer` que usaremos a lo largo del lab combina:

- **`CountVectorizer`** con `StemmerTokenizer`, `ngram_range=(1,2)` y `max_features=500` → aplicado sobre la columna `history_text`.
- **`MinMaxScaler`** → aplicado sobre los 6 atributos numéricos de habilidad: `intelligence_score`, `strength_score`, `speed_score`, `durability_score`, `power_score`, `combat_score`.

In [6]:
preprocessing_transformer = ColumnTransformer(
    transformers=[
        (
            "MinMaxScaler",
            MinMaxScaler(),
            [
                "intelligence_score",
                "strength_score",
                "speed_score",
                "durability_score",
                "power_score",
                "combat_score",
            ],
        ),
        (
            "bow",
            CountVectorizer(
                tokenizer=StemmerTokenizer(),
                max_features=500,
                ngram_range=(1, 2),
            ),
            "history_text",
        ),
    ]
)

## 1.2 Diseño de Baseline y Primer Entrenamiento [1 Punto]

<p align="center">
  <img src="https://pa1.narvii.com/6374/9eaec1b7bf9157334151452a669516f9a78b954c_hq.gif" width="300">
</p>

### 1.2.1 ¿Qué es un Baseline? [0.2 Puntos]

Antes de entrenar modelos complejos, es fundamental establecer un punto de referencia mínimo. Responde las siguientes preguntas con tus propias palabras:

1. **¿Qué es un baseline en Machine Learning?** ¿Para qué sirve establecerlo antes de evaluar modelos más sofisticados?
2. **¿Por qué usamos un `DummyClassifier` como baseline?** ¿Qué implica que un modelo "real" no logre superar su rendimiento?

**Respuesta:**
>* Un baseline es un modelo de referencia mínimo que establece el piso de rendimiento esperado. Su utilidad está en proveer un punto de comparación objetivo: sin él, no es posible determinar si las mejoras de un modelo más sofisticado realmente mejoran o simplemente corresponde a ruido.

>* El DummyClassifier implementa ese baseline en la práctica ignorando completamente las features de entrada y genera salidas aleatorias como respuesta a las predicciones. Que un modelo real no logre superarlo implica que no está extrayendo información útil de los datos, lo que sugiere problemas en el preprocesamiento, la arquitectura del modelo, o incluso en los datos mismos, pues estaría prediciendo peor que el azar.

---

### 1.2.2 Implementación [0.6 Puntos]

Genere un `Pipeline` con las características de 1.1 y un `DecisionTreeClassifier()` por defecto.

Separe el dataset en entrenamiento/prueba (80/20, estratificado, `random_state=RANDOM_STATE`). Entrene, reporte `classification_report` y compare con un `DummyClassifier(strategy="stratified")`.

**To-do:**
- [ ] Pipeline con preprocesamiento → `DecisionTreeClassifier`.
- [ ] Holdout estratificado 80/20.
- [ ] `classification_report` del baseline.
- [ ] Entrenar `DummyClassifier` y comparar.

In [ ]:
# Se definen las variables predictoras y la variable objetivo
X = df_comics.drop(columns=["alignment"])
y = df_comics["alignment"]

# holdout 80/20, estratificado, con random_state=RANDOM_STATE
X_train, X_test, y_train, y_test = train_test_split(
    X, y, shuffle=True, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Se genera el pipeline con el preprocesamiento y el clasificador
DTree_pipe = Pipeline([("preprocesamiento", preprocessing_transformer), ("clasificador", DecisionTreeClassifier())])

# Entrenar el modelo con el pipeline completo
DTree_pipe.fit(X_train, y_train)

# Reporte base de métricas de desempeño en el test como referencia para todas las comparaciones
_DTree_report = pd.DataFrame(
    classification_report(y_test, DTree_pipe.predict(X_test), output_dict=True)
).T[["precision", "recall", "f1-score"]]
_DTree_report

,precision,recall,f1-score
Bad,0.406250,0.453488,0.428571
Good,0.653846,0.574324,0.611511
Neutral,0.129032,0.173913,0.148148
accuracy,0.498054,0.498054,0.498054
macro avg,0.396376,0.400575,0.396077
weighted avg,0.524025,0.498054,0.508825


In [ ]:
# Se genera un pipeline con el preprocesamiento y un clasificador Dummy
dummy_pipe = Pipeline(
    [
        ("preprocesamiento", preprocessing_transformer),
        ("Dummy", DummyClassifier(strategy="stratified")),
    ]
)

# Se entrena el pipeline con el clasificador Dummy
dummy_pipe.fit(X_train, y_train)

# Reporte base de métricas de desempeño en el test del clasificador Dummy
_dummy_report = pd.DataFrame(
    classification_report(y_test, dummy_pipe.predict(X_test), output_dict=True)
).T[["precision", "recall", "f1-score"]]
_dummy_report


,precision,recall,f1-score
Bad,0.329412,0.325581,0.327485
Good,0.594771,0.614865,0.604651
Neutral,0.105263,0.086957,0.095238
accuracy,0.470817,0.470817,0.470817
macro avg,0.343149,0.342468,0.342458
weighted avg,0.462166,0.470817,0.466314


In [34]:
# Comparación de métricas entre el clasificador Dummy y el Decision Tree
_comp = pd.concat(
    [_dummy_report, _DTree_report],
    axis=1,
    keys=["Dummy", "DTree"]
)

_comp

Dummy                         DTree                    
             precision    recall  f1-score precision    recall  f1-score
Bad           0.329412  0.325581  0.327485  0.406250  0.453488  0.428571
Good          0.594771  0.614865  0.604651  0.653846  0.574324  0.611511
Neutral       0.105263  0.086957  0.095238  0.129032  0.173913  0.148148
accuracy      0.470817  0.470817  0.470817  0.498054  0.498054  0.498054
macro avg     0.343149  0.342468  0.342458  0.396376  0.400575  0.396077
weighted avg  0.462166  0.470817  0.466314  0.524025  0.498054  0.508825

### 1.2.3 Pregunta de Cierre [0.2 Puntos]

**Pregunta:** ¿El `DecisionTreeClassifier` supera al `DummyClassifier`? ¿Qué concluyes de esto sobre lo que ha aprendido el modelo? Además responde:

> Sí, el `DecisionTreeClassifier` supera al `DummyClassifier` en todas las métricas. Sin embargo, los márgenes son bastante reducidos: el F1-score macro mejora de 0.342 a 0.396. Esto indica que el modelo está aprendiendo alguna señal de los datos, pero de forma muy débil. En particular, el rendimiento sobre la clase neutral sigue siendo muy bajo (F1 = 0.148), lo que sugiere que el árbol no logra capturar patrones suficientemente discriminativos, especialmente para las clases minoritarias. 

1. ¿Por qué el accuracy puede ser una métrica engañosa en este problema? ¿Qué métrica es más apropiada si las clases están desbalanceadas?
2. ¿Por qué se usa el parámetro `stratify` en el `train_test_split`? ¿Qué problema evitamos al usarlo?
3. ¿Es mejor el clasificador que su versión aleatoria? ¿Podemos avanzar con confianza de que estamos clasificando mejor que si por ejemplo, tiraramos un dado con 3 caras?

Respuesta:
>El accuracy mide la proporción de predicciones correctas que realiza un modelo con respecto al total de predicciones evaluadas. Se calcula mediante la siguiente formula: 
$El accuracy mide la proporción de predicciones correctas sobre el total de predicciones evaluadas. Sin embargo, puede ser una métrica engañosa cuando las clases están desbalanceadas. En este dataset, la clase neutral es considerablemente menos frecuente que good o bad, lo que podría generar un desbalance que el accuracy no captura. Por ejemplo, si el 99% de los personajes fueran héroes, el modelo solo aprendería de la varianza de la clase mayoritaria y omitiría el ruido de la clase minoritaria, por lo tanto, el modelo clasificara a todos como héroes, y obtendría un 99% de accuracy sin haber aprendido nada útil: sería incapaz de detectar un solo villano. En contextos de desbalance, métricas como el F1-score macro o el ROC-AUC son más apropiadas, pues evalúan el rendimiento de forma equitativa sobre todas las clases independientemente de su frecuencia.  

>El parámetro stratify fuerza a que la partición entre train y test respete las mismas proporciones de cada clase que existen en el dataset original, realizando un muestreo estratificado. Sin él, la división aleatoria podría generar conjuntos con distribuciones de clases muy distintas entre sí. Esto es importante cuando hay clases minoritarias, sin estratificación, podría ocurrir que dicha clase quede subrepresentada o ausente en train, impidiendo que el modelo aprenda a reconocerla, o ausente en test, haciendo que la evaluación no sea representativa del problema real. 

>Aunque el `DecisionTreeClassifier` supera al `DummyClassifier`, la diferencia es marginal y no permite avanzar con confianza. En ese sentido, el modelo sí mejora respecto a como clasifica el azar, pero con un F1-score macro de 0.396 y un rendimiento particularmente deficiente en las clases minoritarias, por lo que, no hay evidencia suficiente de que esté capturando estructura real en los datos. 


---

# 2. Métodos de Ensamblaje [2 Puntos]

<p align="center">
  <img src="https://media.giphy.com/media/l0HlHFRbmaZtBRhXG/giphy.gif" width="300">
</p>

Los métodos de ensamblaje combinan múltiples modelos para obtener predicciones más robustas. Exploraremos tres estrategias:

| Estrategia | Idea clave | Ejemplo |
|------------|-----------|---------|
| **Bagging** | Modelos en paralelo sobre subconjuntos aleatorios | Random Forest |
| **Boosting** | Modelos en secuencia, cada uno corrige al anterior | XGBoost, LightGBM |
| **Stacking** | Predicciones de modelos base como input de un meta-modelo | StackingClassifier |

Todos usarán el mismo `preprocessing_transformer` de la sección 1.

## 2.1 Bagging: Random Forest [0.5 Puntos]

### 2.1.1 Descripción del algoritmo [0.2 Puntos]

Describe con tus propias palabras cómo funciona el **Bagging (Bootstrap Aggregating)**. La descripción debe cubrir los siguientes tres pasos:

1. **Generación de subconjuntos**: ¿Cómo se obtienen los subconjuntos de entrenamiento a partir del dataset original? ¿Se usa todo el dataset en cada uno? ¿Se pueden repetir instancias?
2. **Entrenamiento**: ¿Qué se entrena sobre cada subconjunto? ¿Los modelos se entrenan de forma dependiente o independiente entre sí?
3. **Agregación**: ¿Cómo se combinan las predicciones de todos los modelos para obtener una respuesta final?

Respuesta:
> El Bagging (Bootstrap Aggregating) es un método de ensamble que opera en tres etapas. Primero, a partir del dataset original se generan múltiples subconjuntos de entrenamiento mediante muestreo con reemplazo (bootstrap): cada subconjunto tiene el mismo tamaño que el original, pero dado que las instancias se seleccionan aleatoriamente y pueden repetirse, cada uno resulta distinto. Segundo, sobre cada uno de estos subconjuntos se entrena un modelo base de forma completamente independiente y en paralelo. Finalmente, las predicciones de todos los modelos se agregan: en clasificación se utiliza votación por mayoría (la clase más votada), y en regresión se promedia el output de todos los modelos.

Fuente: IBM. (s. f.). ¿En qué consiste el bagging?. _IBM Think_. https://www.ibm.com/es-es/think/topics/bagging 

---

### 2.1.2 Implementación [0.2 Puntos]

**To-do:**
- [ ] Pipeline con `RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)`.
- [ ] Entrenar y reportar `classification_report`.

In [40]:
# Se genera el pipeline con el preprocesamiento y el clasificador RandomForest
RForest_pipe = Pipeline([("preprocesamiento", preprocessing_transformer), ("clasificador", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))])

# Entrenar el modelo con el pipeline completo
RForest_pipe.fit(X_train, y_train)

# Reporte base de métricas de desempeño en el test como referencia para todas las comparaciones
_RForest_report = pd.DataFrame(
    classification_report(y_test, RForest_pipe.predict(X_test), output_dict=True)
).T[["precision", "recall", "f1-score"]]
_RForest_report

,precision,recall,f1-score
Bad,0.666667,0.348837,0.458015
Good,0.657143,0.932432,0.770950
Neutral,0.000000,0.000000,0.000000
accuracy,0.653696,0.653696,0.653696
macro avg,0.441270,0.427090,0.409655
weighted avg,0.601519,0.653696,0.597237


In [41]:
# Comparación de métricas entre el clasificador Dummy y el Random Forest
_comp2 = pd.concat(
    [_dummy_report, _RForest_report],
    axis=1,
    keys=["Dummy", "RForest"]
)

_comp2

Dummy                       RForest                    
             precision    recall  f1-score precision    recall  f1-score
Bad           0.329412  0.325581  0.327485  0.666667  0.348837  0.458015
Good          0.594771  0.614865  0.604651  0.657143  0.932432  0.770950
Neutral       0.105263  0.086957  0.095238  0.000000  0.000000  0.000000
accuracy      0.470817  0.470817  0.470817  0.653696  0.653696  0.653696
macro avg     0.343149  0.342468  0.342458  0.441270  0.427090  0.409655
weighted avg  0.462166  0.470817  0.466314  0.601519  0.653696  0.597237

### 2.1.3 Pregunta de Cierre [0.1 Puntos]

**Pregunta:** ¿El Random Forest mejoró respecto al baseline? Comenta los resultados observados en el `classification_report` y explica a qué se debe la diferencia (o falta de ella), considerando las características del algoritmo que describiste anteriormente.

Respuesta:
> Random Forest mejora respecto al baseline de forma clara en la mayoría de las clases: el F1-score macro aumenta de 0.342 a 0.396, lo que indica una mayor capacidad discriminativa general del modelo. Esta mejora se explica por el mecanismo de bagging combinado con selección aleatoria de features, que reduce la varianza y genera modelos base más diversos que el árbol individual. Sin embargo, el rendimiento sobre la clase neutral es más que deficiente, es peor que el azar. Esto se debe a que el muestreo bootstrap con reemplazo tiende a subrepresentar las clases minoritarias en cada subconjunto, y dado que la predicción final se determina por votación de mayoría, la clase neutral queda  opacada por las clases más frecuentes. En consecuencia, el modelo aprende a ignorarla en lugar de aprenderla. 

## 2.2 Boosting: XGBoost y LightGBM [0.8 Puntos]

### 2.2.1 Descripción del algoritmo [0.3 Puntos]

Describe con tus propias palabras cómo funciona el **Boosting**. Tu descripción debe cubrir los siguientes tres pasos:

1. **Entrenamiento secuencial**: ¿En qué se diferencia el Boosting del Bagging en cuanto al orden en que se entrenan los modelos? ¿Son independientes entre sí?
2. **Corrección de errores**: ¿Cómo sabe cada modelo nuevo en qué instancias debe enfocarse? ¿Qué información del modelo anterior utiliza?
3. **Predicción final**: ¿Cómo se combinan las predicciones de todos los modelos? ¿Es una votación simple o una combinación ponderada?

Además, explica brevemente en qué se diferencian **XGBoost** y **LightGBM** como implementaciones de Boosting, y por qué XGBoost requiere que las etiquetas sean numéricas mientras que LightGBM acepta strings directamente.

Respuesta:
> A diferencia del bagging, donde los modelos base se entrenan en paralelo de forma independiente, el boosting entrena los modelos de forma secuencial: cada modelo nuevo depende del anterior, pues su objetivo es corregir los errores que este cometió.
El mecanismo de corrección varía según la variante. En AdaBoost, cada modelo nuevo recibe más peso sobre las instancias que el modelo anterior clasificó incorrectamente, forzando al siguiente aprendiz a enfocarse en los casos más difíciles. En Gradient Boosting, en cambio, cada modelo se entrena sobre los residuos del anterior, es decir, sobre la diferencia entre la predicción actual y el valor real, aproximando el gradiente del error.
La predicción final no es una votación simple, sino una combinación ponderada de todos los modelos base: cada modelo contribuye proporcionalmente a su desempeño, de modo que los modelos más precisos tienen mayor influencia en la decisión final.
XGBoost y LightGBM son dos implementaciones de gradient boosting que difieren principalmente en eficiencia. XGBoost construye los árboles nivel por nivel, mientras que LightGBM los construye hoja por hoja, lo que lo hace más rápido y eficiente en memoria para datasets grandes. Respecto a las etiquetas, XGBoost requiere que sean numéricas porque su función de pérdida opera directamente sobre valores numéricos para calcular gradientes y hessianos. LightGBM, en cambio, incorpora un codificador interno que transforma las etiquetas de texto automáticamente antes del entrenamiento, por lo que acepta strings directamente sin preprocesamiento adicional.

Fuente: IBM. (s. f.). ¿Qué es el boosting de gradiente?. _IBM Think_. https://www.ibm.com/es-es/think/topics/gradient-boosting

---

### 2.2.2 Implementación [0.4 Puntos]

**To-do:**
- [ ] Crear `LabelEncoder`, ajustarlo sobre `y_train` y transformar `y_train` e `y_test`.
- [ ] Pipeline con `XGBClassifier(random_state=RANDOM_STATE, eval_metric="mlogloss", verbosity=0)`. Reportar resultados decodificando las predicciones con `le.inverse_transform`.
- [ ] Pipeline con `LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)`. Reportar resultados.

In [ ]:
# Crear Label Encoder
le = LabelEncoder()

# Ajusarlo sobre train y transformar y_train e y_test
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# Se genera el pipeline con el preprocesamiento y el clasificador XBGClassifier
XBG_pipe = Pipeline([("preprocesamiento", preprocessing_transformer), ("clasificador", XGBClassifier(random_state=RANDOM_STATE, eval_metric="mlogloss", verbosity=0))])

# Entrenar el modelo con el pipeline completo
XBG_pipe.fit(X_train, y_train_encoded)

# Función para decodificar las etiquetas predichas por el modelo
def decode_labels(encoded_labels):
    return le.inverse_transform(encoded_labels)

# Reporte base de métricas de desempeño en el test como referencia para todas las comparaciones
_XBG_report = pd.DataFrame(
    classification_report(y_test_encoded, XBG_pipe.predict(X_test), output_dict=True)
).T[["precision", "recall", "f1-score"]]

# Decodificar las etiquetas en el índice del reporte
_XBG_report.index = [
    decode_labels([int(i)])[0] if str(i).isdigit() else i
    for i in _XBG_report.index
]
_XBG_report


,precision,recall,f1-score
Bad,0.562500,0.418605,0.480000
Good,0.675676,0.844595,0.750751
Neutral,0.500000,0.173913,0.258065
accuracy,0.642023,0.642023,0.642023
macro avg,0.579392,0.479037,0.496272
weighted avg,0.622082,0.642023,0.616057


In [47]:
# Se genera el pipeline con el preprocesamiento y el clasificador LGBMClassifier
LGBM_pipe = Pipeline([("preprocesamiento", preprocessing_transformer), ("clasificador", LGBMClassifier(random_state=RANDOM_STATE, verbose=-1))])

# Entrenar el modelo con el pipeline completo
LGBM_pipe.fit(X_train, y_train)

# Reporte base de métricas de desempeño en el test como referencia para todas las comparaciones
_LGBM_report = pd.DataFrame(
    classification_report(y_test, LGBM_pipe.predict(X_test), output_dict=True)
).T[["precision", "recall", "f1-score"]]

_LGBM_report

,precision,recall,f1-score
Bad,0.600000,0.558140,0.578313
Good,0.717514,0.858108,0.781538
Neutral,0.000000,0.000000,0.000000
accuracy,0.680934,0.680934,0.680934
macro avg,0.439171,0.472083,0.453284
weighted avg,0.613977,0.680934,0.643590


In [48]:
# Comparación de métricas entre el clasificador Dummy, XBGClassifier y LGBMClassifier
_comp3 = pd.concat(
    [_dummy_report, _XBG_report, _LGBM_report],
    axis=1,
    keys=["Dummy", "XBG", "LGBM"]
)

_comp3

Dummy                           XBG                      \
             precision    recall  f1-score precision    recall  f1-score   
Bad           0.329412  0.325581  0.327485  0.562500  0.418605  0.480000   
Good          0.594771  0.614865  0.604651  0.675676  0.844595  0.750751   
Neutral       0.105263  0.086957  0.095238  0.500000  0.173913  0.258065   
accuracy      0.470817  0.470817  0.470817  0.642023  0.642023  0.642023   
macro avg     0.343149  0.342468  0.342458  0.579392  0.479037  0.496272   
weighted avg  0.462166  0.470817  0.466314  0.622082  0.642023  0.616057   

                  LGBM                      
             precision    recall  f1-score  
Bad           0.600000  0.558140  0.578313  
Good          0.717514  0.858108  0.781538  
Neutral       0.000000  0.000000  0.000000  
accuracy      0.680934  0.680934  0.680934  
macro avg     0.439171  0.472083  0.453284  
weighted avg  0.613977  0.680934  0.643590

### 2.2.3 Pregunta de Cierre [0.1 Puntos]

**Pregunta:** Compara los resultados de XGBoost y LightGBM según el `classification_report`. ¿Cuál tuvo mejor desempeño en F1-Macro? ¿Ambos mejoran respecto al baseline? Considerando las diferencias que describiste en 2.2.1, ¿a qué atribuyes las similitudes o diferencias en rendimiento?

Respuesta:
>Ambos modelos mejoran respecto al baseline de forma notable: XGBoost obtiene un F1-macro de 0.496 y LightGBM de 0.453, frente al 0.342 del DummyClassifier. Sin embargo, XGBoost supera a LightGBM en F1-macro, principalmente por su mejor manejo de la clase minoritaria (neutral). De hecho, LightGBM no logra identificar ninguna instancia 'neutral' correctamente, siendo superado incluso por el clasificador aleatorio.
Esta diferencia puede atribuirse al mecanismo de crecimiento de árboles: LightGBM construye sus árboles hoja por hoja priorizando las hojas con mayor ganancia, y en un dataset desbalanceado como este, esas ganancias tienden a concentrarse en las clases mayoritarias. Como resultado, el modelo sesga su aprendizaje hacia 'good' y 'bad', ignorando sistemáticamente 'neutral'. XGBoost, al construir nivel por nivel, distribuye el aprendizaje de forma más uniforme entre clases, lo que le permite capturar aunque sea parcialmente la clase minoritaria.

## 2.3 Stacking [0.7 Puntos]

### 2.3.1 Descripción del algoritmo [0.2 Puntos]

Describe con tus propias palabras cómo funciona el **Stacking**. Tu descripción debe cubrir los siguientes tres aspectos:

1. **Predicciones como features**: ¿Qué rol cumplen los modelos base? ¿Sobre qué datos generan sus predicciones para ser usadas por el meta-modelo? ¿Por qué se usa validación cruzada interna en lugar de predecir directamente sobre los datos de entrenamiento?
2. **Meta-modelo**: ¿Qué recibe como input el meta-modelo y qué aprende? ¿En qué se diferencia su rol del de los modelos base?
3. **Ventaja sobre selección simple**: ¿Por qué el Stacking puede superar a cualquier modelo base individual? ¿Qué aprovecha de la diversidad entre modelos?

> **Respuesta:**

In [ ]:
# Escribe aquí la respuesta

---

### 2.3.2 Implementación [0.4 Puntos]

**Restricciones:**
- Mínimo **3 modelos base distintos**.
- Solo clasificadores básicos de scikit-learn: `LogisticRegression`, `MultinomialNB`, `SGDClassifier`, `DecisionTreeClassifier`, etc. **No se permiten modelos de ensamblaje** (`RandomForest`, `XGBoost`, `LightGBM`).
- El meta-modelo es de libre elección. Justifica tu elección.

**To-do:**
- [ ] Definir al menos 3 modelos base (solo clasificadores básicos de scikit-learn).
- [ ] Elegir un meta-modelo.
- [ ] Pipeline con `StackingClassifier(cv=3, n_jobs=-1)`.
- [ ] Reportar `classification_report`.

In [ ]:
#### Código aquí ####

### 2.3.3 Pregunta de Cierre [0.1 Puntos]

**Pregunta:** Justifica la elección de tus modelos base y meta-modelo: ¿por qué los elegiste y qué aporta cada uno? ¿El Stacking mejoró respecto a los modelos base individuales de las secciones anteriores? ¿A qué atribuyes ese resultado considerando cómo funciona el algoritmo?

> **Respuesta:**

In [ ]:
# Escribe aquí la respuesta

---

# 3. Optimización de Hiperparámetros con Optuna [1.5 Puntos]

<p align="center">
  <img src="https://media.giphy.com/media/3oKIPEqDGUULpEU0aQ/giphy.gif" width="300">
</p>

Hasta ahora hemos usado hiperparámetros por defecto. En esta sección optimizaremos **LightGBM** con **Optuna**, una librería de optimización bayesiana más eficiente que `GridSearchCV` porque:

- **Samplers inteligentes (TPE)**: usa el historial de trials para proponer mejores valores.
- **Pruning**: abandona trials poco prometedores antes de completarlos.
- **Visualizaciones interactivas**: analiza el espacio de hiperparámetros con `optuna-dashboard`.

Los resultados del estudio se persistirán en una base de datos SQLite, lo que permite explorarlos en tiempo real con el dashboard.

### 3.1 Descripción: Hiperparámetros y Optimización [0.2 Puntos]

Antes de implementar la optimización, reflexiona sobre los siguientes conceptos:

1. **¿Qué son los hiperparámetros?** ¿En qué se diferencian de los parámetros que el modelo aprende durante el entrenamiento?
2. **¿Por qué es importante optimizarlos?** ¿Qué consecuencias puede tener usar hiperparámetros por defecto?
3. **¿Qué es `GridSearchCV`?** Describe cómo funciona su mecanismo de búsqueda y cuál es su principal limitación cuando el espacio de hiperparámetros es grande.
4. **¿Por qué se evalúa cada trial con validación cruzada en lugar de usar directamente el conjunto de test?** ¿Qué problema introduce usar el test para seleccionar hiperparámetros? ¿Cómo podríamos reemplazar la CV por un conjunto de validación separado, y cuáles serían las ventajas y desventajas de ese enfoque?
5. **¿Cómo mejoran los samplers inteligentes la limitación de `GridSearchCV`?** ¿Por qué un sampler como TPE puede ser más eficiente que una búsqueda exhaustiva?

> **Respuesta:**

In [ ]:
# Escribe aquí la respuesta

### 3.2 Definición del Espacio de Búsqueda y Función Objetivo [0.5 Puntos]

La función `objective(trial)` recibe un trial de Optuna, define los hiperparámetros a probar y devuelve la métrica a optimizar (F1-Macro).

| Hiperparámetro | Tipo | Rango | Descripción |
|----------------|------|-------|-------------|
| `num_leaves` | `suggest_int` | [20, 200] | Número máximo de hojas por árbol; controla la complejidad del modelo. |
| `learning_rate` | `suggest_float` (log) | [1e-3, 0.3] | Tasa de aprendizaje; cuánto contribuye cada árbol a la predicción final. |
| `max_depth` | `suggest_int` | [3, 12] | Profundidad máxima de cada árbol; limita la capacidad de memorización. |
| `min_child_samples` | `suggest_int` | [5, 100] | Mínimo de muestras requeridas en una hoja; regulariza contra sobreajuste. |
| `subsample` | `suggest_float` | [0.5, 1.0] | Fracción de filas muestreadas aleatoriamente para entrenar cada árbol. |
| `colsample_bytree` | `suggest_float` | [0.5, 1.0] | Fracción de features muestreadas aleatoriamente para cada árbol. |
| `reg_alpha` | `suggest_float` (log) | [1e-8, 10.0] | Regularización L1 sobre los pesos de las hojas. |
| `reg_lambda` | `suggest_float` (log) | [1e-8, 10.0] | Regularización L2 sobre los pesos de las hojas. |

**To-do:**
- [ ] Implementar `objective(trial)` con el espacio de búsqueda indicado.
- [ ] Usar `StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)`.
- [ ] Retornar la media del F1-Macro sobre los 3 folds.

In [ ]:
#### Código aquí ####

In [ ]:
def objective(trial):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 20, 200),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }

    pipe = Pipeline(
        [
            ("features", preprocessing_transformer),
            ("clf", LGBMClassifier(**params, random_state=RANDOM_STATE, verbose=-1)),
        ]
    )

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1_macro")
    return scores.mean()

### 3.3 Ejecución del Estudio de Optimización [0.4 Puntos]

Crea el estudio Optuna con 50 trials usando `TPESampler`. El estudio se guarda en el storage SQLite configurado anteriormente para poder explorarlo con `optuna-dashboard`.

**To-do:**
- [ ] Crear el estudio con `direction="maximize"`, `TPESampler(seed=RANDOM_STATE)` y el `storage` configurado.
- [ ] Ejecutar `study.optimize` con `n_trials=50` y `show_progress_bar=True`
- [ ] Imprimir el mejor valor y los mejores hiperparámetros.

In [ ]:
storage = optuna.storages.RDBStorage("sqlite:///optuna_lab7.db")
study = optuna.create_study(
    direction="maximize",
    study_name="lgbm-comics",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    storage=storage,
    load_if_exists=True,
)

study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"Mejor F1-Macro: {study.best_value:.4f}")
print("\nMejores hiperparámetros:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

### 3.4 Visualizaciones de Optuna [0.2 Puntos]

Optuna provee visualizaciones interactivas para analizar el proceso de optimización.

**To-do:**
- [ ] Graficar el historial de optimización (`plot_optimization_history`).
- [ ] Graficar la importancia de hiperparámetros (`plot_param_importances`).
- [ ] Graficar una visualización del comportamiento de los hiperparámetros (`plot_parallel_coordinate`)

In [ ]:
#### Código aquí ####

### 3.5 Comparar Modelo Base con Mejores Hiperparámetros [0.1 Puntos]

**To-do:**
- [ ] Reentrenar un pipeline `pipe_lgbm_opt` con `study.best_params`.
- [ ] Comparar F1-Macro entre `pipe_lgbm` (defaults) y `pipe_lgbm_opt` (optimizado).

In [ ]:
#### Código aquí ####

### 3.6 Preguntas de reflexión [0.1 Puntos]

1. Según `plot_param_importances`, ¿qué hiperparámetro tuvo más impacto? ¿Tiene sentido dado tu conocimiento sobre algoritmos de Boosting?
2. ¿Qué información adicional te entregaron los gráficos de optuna comparado con solo mirar los logs de texto del estudio? ¿Qué pasa si en el gráfico interactivo `plot_parallel_coordinate` seleccionas con el mouse el rango específico con mejores resultados, hay patrones interesantes? - Hint: hace drag con el puntero sobre la dimensión de _Objective Value_ para seleccionar las mejores observaciones.

In [ ]:
# Escribe aquí la respuesta

# 4. Interpretabilidad [1 Punto]

<p align="center">
  <img src="https://media.giphy.com/media/xT9IgzoKnwFNmISR8I/giphy.gif" width="300">
</p>

En esta sección interpretaremos las predicciones de LightGBM. Para obtener interpretaciones claras y directas, entrenaremos un **modelo auxiliar de LGBM usando únicamente los 6 atributos numéricos** (scores de habilidades), sin incluir features de texto.

Esto nos permite:
- Aplicar PFI y SHAP sin complicaciones de alta dimensionalidad.
- Obtener explicaciones semánticamente significativas (fuerza, inteligencia, velocidad...).

> **Nota:** En esta sección entrenaremos un modelo auxiliar para interpretabilidad con menos atributos. Usaremos un modelo entrenado desde 0, no el mejor seleccionado anteriormente.

In [ ]:
NUMERICAL_FEATURES = [
    "intelligence_score",
    "strength_score",
    "speed_score",
    "durability_score",
    "power_score",
    "combat_score",
]

X_train_num = X_train[NUMERICAL_FEATURES]
X_test_num = X_test[NUMERICAL_FEATURES]

# Reentrenar LGBM con los mejores hiperparámetros, solo sobre scores numéricos
lgbm_interp = ...

## 4.1 Permutation Feature Importance (PFI) [0.4 Puntos]

### 4.1.1 Descripción [0.1 Puntos]

Responde las siguientes preguntas con tus propias palabras:

1. **¿Cómo funciona la Permutation Feature Importance?** ¿Qué se permuta exactamente y cómo se mide el impacto en el rendimiento del modelo?
2. **¿Por qué PFI es preferible a la importancia nativa de los árboles de decisión?** ¿Qué sesgo tiene la importancia nativa que PFI evita?

> **Respuesta:**

In [ ]:
# Escribe aquí la respuesta

---

### 4.1.2 Implementación [0.2 Puntos]

**To-do:**
- [ ] Calcular `permutation_importance` con `n_repeats=30` y `scoring="f1_macro"`.
- [ ] Graficar un boxplot horizontal con plotly (`px.bar` con `orientation='h'`) con la importancia y varianza de cada feature.

In [ ]:
#### Código aquí ####

### 4.1.3 Pregunta de Cierre [0.1 Puntos]

**Pregunta:** Según el gráfico de PFI, ¿qué feature tuvo mayor importancia? ¿Tiene sentido intuitivo dado el contexto del problema (clasificar la alineación de personajes de cómics)? ¿Hay alguna feature cuya importancia te sorprenda?

> **Respuesta:**

In [ ]:
# Escribe aquí la respuesta

## 4.2 SHAP (SHapley Additive exPlanations) [0.6 Puntos]

### 4.2.1 Descripción [0.2 Puntos]

Responde las siguientes preguntas con tus propias palabras:

1. **¿Qué miden los SHAP values?** ¿En qué se diferencian de una medida de importancia global como PFI?
2. **¿Qué representa el `base_value` en un `waterfall_plot` de SHAP?** ¿Por qué es el punto de partida para interpretar una predicción individual?
3. **¿Por qué usar `TreeExplainer` para modelos basados en árboles?** ¿Qué ventaja ofrece respecto a un explainer genérico?

> **Respuesta:**

In [ ]:
# Escribe aquí la respuesta

---

### 4.2.2 Implementación [0.2 Puntos]

**To-do:**
- [ ] Crear un `shap.TreeExplainer(lgbm_interp)` y calcular `shap_values` sobre `X_test_num`.
- [ ] `summary_plot` para ver importancia global y dirección del efecto (para la clase "Good").
- [ ] `waterfall_plot` para la predicción de una instancia cualquiera de `X_test_num`.

In [ ]:
#### Código aquí ####

### 4.2.3 Pregunta de Cierre [0.2 Puntos]

1. ¿Qué diferencia existe entre Permutation Feature Importance y los SHAP values como medida de importancia de features?
2. Según el `waterfall_plot`, ¿qué features fueron las que más empujaron la predicción hacia su clase? Investiga el personaje seleccionado: ¿Tiene sentido dado su historia en los cómics?

> **Respuesta:**

In [ ]:
# Escribe aquí la respuesta

---

# 5. Predicción de Personajes No Etiquetados [0.5 Puntos]

<p align="center">
  <img src="https://pbs.twimg.com/media/DolotxUUYAAbg7f.jpg" width="350">
</p>

¡Llegó el momento de predecir `Vergil`, `Gorilla Girl` y `Bat-Cow`!

Usaremos el **mejor modelo** obtenido en la sección 3 (`pipe_lgbm_opt`) para predecir la alineación de los personajes no etiquetados.

**Nota:** Recuerda eliminar los NaN en `history_text` antes de predecir.

### 5.0 Predicción [0.2 Puntos]

**To-do:**
- [ ] Usar `pipe_lgbm_opt` para predecir `alignment` en `df_comics_no_label` (recuerda eliminar NaN en `history_text`).
- [ ] Filtrar y mostrar resultados para `Vergil`, `Gorilla Girl` y `Bat-Cow`.

In [ ]:
#### Código aquí ####

### 5.1 Análisis de Predicciones [0.3 Puntos]

**Pregunta:** Comenta las predicciones obtenidas para `Vergil`, `Gorilla Girl` y `Bat-Cow`:

1. ¿Las predicciones te parecen razonables según lo que conoces (o puedes inferir) de estos personajes?
2. Conecta con la sección 4: ¿qué features numéricas habrían influido más en la predicción de **Bat-Cow** según el `waterfall_plot`? ¿Es consistente con la predicción obtenida aquí?

> **Respuesta:**

In [ ]:
# Escribe aquí la respuesta

# Conclusión

¡Eso ha sido todo para el lab de hoy! Recuerden que el laboratorio tiene un plazo de entrega de una semana y que **los días de atraso no se pueden utilizar para entregas de lab, solo para tareas**. Cualquier duda del laboratorio, no duden en contactarnos por mail o U-cursos.

<p align="center">
  <img src="https://media1.tenor.com/images/fb5bf7cc5a4acb91b4177672886a88ba/tenor.gif?itemid=5591338">
</p>